In [ ]:
import pandas as pd
from pathlib import Path

interim_path = Path("../data/interim")


In [ ]:
routes = pd.read_csv(interim_path/"routes.csv")
stops = pd.read_csv(interim_path/"stops.csv")
validations = pd.read_csv(interim_path/"validations.csv")
vehicles = pd.read_csv(interim_path/"vehicles.csv")

print("All four interim datasets loaded successfully")

All four interim datasets loaded successfully


## Cleaning the Validations File


#### The validation data will be checked for both completely duplicated rows and repated "validation_id" values

In [ ]:
print("Exact duplicate rows:", validations.duplicated().sum())
print("Duplicate validation_id:",validations["validation_id"].duplicated().sum())


Exact duplicate rows: 802
Duplicate validation_id: 1600


### Remove duplicate validations
#### The cleaning function will remove exact duplicate rows first and then remove repeated "validation_id" values while keeping the first occurrence 

In [ ]:
def remove_validation_duplicates(df):
    df = df.drop_duplicates()
    df = df.drop_duplicates(subset="validation_id",keep= "first")
    return df 


In [ ]:
before = len(validations)
validations = remove_validation_duplicates(validations)

after =len(validations)

print("Rows dropped:", before - after)
print("Rows remaining:", after)


Rows dropped: 1600
Rows remaining: 80000


In [ ]:
validations["validation_ts"].head(20)


0       2024-05-13 18:54:29
1          06/06/2024 08:48
2          06/03/2024 13:15
3       2024-04-21 17:30:00
4       2024-06-14 18:35:27
5       2024-02-16 18:30:59
6          03/03/2024 08:41
7       2024-04-14 17:01:25
8       2024-03-09 11:02:00
9          03/06/2024 08:08
10    March 9, 2024 6:00 PM
11      2024-03-08 19:29:42
12      2024-02-13 13:08:10
13         13/02/2024 17:35
14      2024-01-21 21:10:12
15      2024-05-16 21:21:17
16         03/04/2024 19:07
17         27/02/2024 12:22
18      2024-04-26 17:32:14
19      2024-04-23 08:29:19
Name: validation_ts, dtype: str

In [ ]:
validations["validation_ts_parsed"] = pd.to_datetime(validations["validation_ts"],format="mixed",dayfirst=True, errors="coerce")



In [ ]:
validations[["validation_ts","validation_ts_parsed"]].head(20)

,validation_ts,validation_ts_parsed
0,2024-05-13 18:54:29,2024-05-13 18:54:29
1,06/06/2024 08:48,2024-06-06 08:48:00
2,06/03/2024 13:15,2024-03-06 13:15:00
3,2024-04-21 17:30:00,2024-04-21 17:30:00
4,2024-06-14 18:35:27,2024-06-14 18:35:27
5,2024-02-16 18:30:59,2024-02-16 18:30:59
6,03/03/2024 08:41,2024-03-03 08:41:00
7,2024-04-14 17:01:25,2024-04-14 17:01:25
8,2024-03-09 11:02:00,2024-09-03 11:02:00
9,03/06/2024 08:08,2024-06-03 08:08:00


In [ ]:
validations["validation_ts_parsed"].isna().sum()

np.int64(0)

In [ ]:
validations["validation_ts"] = validations["validation_ts_parsed"]
validations["validation_ts"] = validations["validation_ts"].astype("datetime64[ns]")


In [ ]:
validations = validations.drop(columns=["validation_ts_parsed"])

In [ ]:
validations["validation_ts"].dtype

dtype('<M8[ns]')

In [ ]:
validations["validation_ts"].head()

0   2024-05-13 18:54:29
1   2024-06-06 08:48:00
2   2024-03-06 13:15:00
3   2024-04-21 17:30:00
4   2024-06-14 18:35:27
Name: validation_ts, dtype: datetime64[ns]

### Strip whitespaces from all ID columns
#### The ID columns in the validations datset will be stripped of leading and trailing whitespace without changing the actual ID values

In [ ]:
validations.columns.tolist()

['validation_id',
 'validation_ts',
 'route_id',
 'vehicle_id',
 'stop_id',
 'card_id',
 'fare_type',
 'fare_amount',
 'passenger_count',
 'transfer_flag']

In [ ]:
id_columns=[c for c in validations.columns if "id" in c.lower()]
id_columns

['validation_id',
 'validation_ts',
 'route_id',
 'vehicle_id',
 'stop_id',
 'card_id']

In [ ]:
def strip_id(df):
    id_columns =[c for c in df.columns if "id" in c.lower()]
    for col in id_columns:
        df[col]=df[col].astype("string").str.strip()
    return df
    validations= strip_id(validations)


In [ ]:
validations[id_columns].head()

,validation_id,validation_ts,route_id,vehicle_id,stop_id,card_id
0,VAL-0068918,2024-05-13 18:54:29,R001,V0044,S0234,CARD-001739
1,VAL-0044034,2024-06-06 08:48:00,R058,V0086,S0131,CARD-000729
2,VAL-0017450,2024-03-06 13:15:00,R043,V0177,S0251,CARD-020205
3,VAL-0004736,2024-04-21 17:30:00,R026,V0108,S0330,CARD-013092
4,VAL-0053735,2024-06-14 18:35:27,R003,V0057,S0389,CARD-003251


#### The fare_type column contains different spellings for the same category. These values are normalized to the fixed set: Adult, Student, Senior, Child and Pass Holder.


In [ ]:
original_values= (validations["fare_type"]
                  .dropna()
                  .astype("string")
                  .str.strip()
                  .str.lower()
                  .unique())

print(sorted(original_values))

['adult', 'child', 'full fare', 'kid', 'monthly pass', 'pass holder', 'passholder', 'senior', 'senior citizen', 'student']


In [ ]:
validations_original= pd.read_csv(interim_path/"validations.csv")

In [ ]:
def normalize_fare_type(df):
    original_fare_type = (validations_original.loc[df.index, "fare_type"]
                          .astype(str)
                          .str.strip()
                          .str.lower()
                          )
    mapping = {
         "adult": "Adult",
         "full fare": "Adult",
        "student": "Student",
        "senior": "Senior",
        "senior citizen": "Senior",
        "kid": "Child",
        "child": "Child",
        "pass holder": "Pass Holder",
        "passholder": "Pass Holder",
        "monthly pass": "Pass Holder"
     }
    df["fare_type"]= original_fare_type.map(mapping)
    return df
    

In [ ]:
validations = normalize_fare_type(validations)

In [ ]:
validations["fare_type"].value_counts(dropna=False)

fare_type
Adult          44401
Student        11847
Senior          9471
Pass Holder     7958
Child           6323
Name: count, dtype: int64

#### The "transfer_flag" column contains multiple representations of the same boolean information. These values will be normalized to a consistent boolean representation. 

In [ ]:
validations["transfer_flag"].value_counts()

transfer_flag
N        36179
Y        24101
0         7906
FALSE     5488
1         3974
TRUE      2352
Name: count, dtype: int64

In [ ]:
def normalize_transfer_flag(df):
    mapping ={
        "Y": "True",
        "N":"False",
        "1": "True",
        "0": "False",
        "TRUE": "True",
        "FALSE": "False"
   }
    df["transfer_flag"]= (
        df["transfer_flag"].astype(str)
        .str.strip()
        .str.upper()
        .map(mapping)
    )
    return df

In [ ]:
validations= normalize_transfer_flag(validations)

In [ ]:
validations["transfer_flag"].value_counts(dropna=False)

transfer_flag
False    49573
True     30427
Name: count, dtype: int64

#### Missing passenger_count values were removed because the passenger count is required for the analysis. Negative passenger count were kept because theu mey represent reversed or corrected transactions. An "Is_reversed" boolean was created to identify these records without removing useful information.

In [ ]:
def handle_passenger_count(df):
    null_count= df["passenger_count"].isna().sum()
    df= df.dropna(subset=["passenger_count"]).copy()
    df["is_reversed"]= df["passenger_count"]< 0
    return df, null_count

In [ ]:
validations, passenger_nulls = handle_passenger_count(validations)
print("Passenger count rows dropped:", passenger_nulls)
print("Rows remaining:", len(validations))

Passenger count rows dropped: 811
Rows remaining: 79189


In [ ]:
print(len(validations))
print(validations["is_reversed"].sum())

79189
358


In [ ]:
validations.to_csv("data/interim/validations.csv", index= False)

In [ ]:
validations[["passenger_count", "is_reversed"]].head(20)

,passenger_count,is_reversed
0,1.0,False
1,1.0,False
2,1.0,False
3,3.0,False
4,1.0,False
5,1.0,False
6,1.0,False
7,1.0,False
8,1.0,False
9,1.0,False


In [ ]:
validations["is_reversed"].value_counts()

is_reversed
False    78831
True       358
Name: count, dtype: int64

#### We filled missing fares with median fare for that fare type, since fare differ a lot between types like Student and Adult.

In [ ]:
validations["fare_amount"].isna().sum()

np.int64(1156)

In [ ]:
validations.groupby("fare_type")["fare_amount"].median()

fare_type
Adult          2.5
Child          1.0
Pass Holder    0.0
Senior         1.2
Student        1.5
Name: fare_amount, dtype: float64

In [ ]:
validations["fare_amount"]= (validations.groupby("fare_type")["fare_amount"].transform(lambda x:x.fillna(x.median())))

In [ ]:
validations["fare_amount"].isna().sum()

np.int64(0)

#### We capped very high fares at 99.5th percentile instead of removing them,so we don't lose data.

In [ ]:
fare_cap =validations["fare_amount"].quantile(0.995)

print("99.5th percentile:", fare_cap)

99.5th percentile: 2.86


In [ ]:
fare_cap

np.float64(2.86)

In [ ]:
(validations["fare_amount"]> fare_cap).sum()

np.int64(362)

In [ ]:
validations["fare_amount"]= validations["fare_amount"].clip(upper=fare_cap)

In [ ]:
validations["fare_amount"].max()

np.float64(2.86)

#### In this step we are replacing the blank card_id into "SINGLE_RIDE"

In [ ]:
def fix_card_id(df):
    df = df.copy()
    blank_count= (df["card_id"].isna() | (df["card_id"].str.strip()=="")).sum()
    df["card_id"]= df["card_id"].fillna("SINGLE_RIDE")
    df.loc[df["card_id"].str.strip()=="","card_id"]="SINGLE_RIDE"
    print(f"Replaced {blank_count} blank card_id values with SINGLE_RIDE")
    return df, blank_count
validations, n_card_id_fixed= fix_card_id(validations)

Replaced 9448 blank card_id values with SINGLE_RIDE


In [ ]:
print(validations["card_id"].isna().sum())
print((validations["card_id"]=="").sum())
print(validations["card_id"].unique()[:20])

0
0
<StringArray>
['CARD-001739', 'CARD-000729', 'CARD-020205', 'CARD-013092', 'CARD-003251',
 'CARD-010226', 'SINGLE_RIDE', 'CARD-004767', 'CARD-002453', 'CARD-008366',
 'CARD-023143', 'CARD-001326', 'CARD-020877', 'CARD-012606', 'CARD-007551',
 'CARD-005721', 'CARD-021303', 'CARD-008107', 'CARD-008876', 'CARD-019805']
Length: 20, dtype: str


## Cleaning the Routes File

#### In this step we have trimmed whitespaces and title-cased "route_name"

In [ ]:
def fix_route_name(df):
    df=df.copy()
    df["route_name"]= df["route_name"].str.strip().str.title()
    return df
routes= fix_route_name(routes)

#### Here we are doing the same method for the column "mode"

In [ ]:
def fix_mode(df):
    df=df.copy()
    df["mode"] = df["mode"].str.strip().str.title()
    return df 
routes= fix_mode(routes)

#### We are filling the null values with "Unknown"

In [ ]:
def fix_district(df):
    df=df.copy()
    null_count= df["district"].isna().sum()
    df["district"]= df["district"].fillna("Unknown")
    print(f"Filled {null_count} null district values with 'Unknown'")
    return df, null_count
routes, n_district_filled = fix_district(routes)


Filled 3 null district values with 'Unknown'


In [ ]:
def fix_route_length(df):
    df= df.copy()
    null_count = df ["route_length_km"].isna().sum()
    medians = df.groupby("mode")["route_length_km"].transform("median")
    df["route_length_km"]= df["route_length_km"].fillna(medians)
    print(f" Filled {null_count} null route_length_km values using per mode median")
    return df, null_count
routes, n_route_length_filled = fix_route_length(routes)

 Filled 3 null route_length_km values using per mode median


## Cleaning the Vehicles File 

In [ ]:
def fix_year_manufactured(df):
    df= df.copy()
    invalid_mask = (df["year_manufactured"]< 1980) | (df["year_manufactured"]> 2026)
    invalid_count= invalid_mask.sum()

    df.loc[invalid_mask, "year_manufactured "]= pd.NA

    print(f"Set {invalid_count} out-of-range year_manufactured values to null")
    return df, invalid_count
vehicles, n_year_fixed = fix_year_manufactured(vehicles)

Set 15 out-of-range year_manufactured values to null


In [ ]:
import numpy as np

In [ ]:
def add_vehicle_age_band(df, current_year=2026):
    df=df.copy()
    age= current_year- df["year_manufactured"]
    conditions=[
        age.isna(),
        age <= 5,
        age <= 10,
        age <= 20,
    ]
    choices= ["Unknown","0-5","6-10","11-20"]
    df["vehicle_age_band"]= np.select(conditions, choices, default= "20+")
    print("Derived vehicle_age_band from year_manufactured")
    return df
vehicles= add_vehicle_age_band(vehicles)

Derived vehicle_age_band from year_manufactured


In [ ]:
def fix_capacity(df):
    df= df.copy()
    null_count = df["capacity"].isna().sum()

    medians= df.groupby("vehicle_type")["capacity"].transform("median")
    df["capacity"]= df["capacity"].fillna(medians)

    print(f"Filled {null_count} null capacity values using per-vehicle_type median")
    return df, null_count
vehicles, n_capacity_filled = fix_capacity(vehicles)


Filled 12 null capacity values using per-vehicle_type median


In [ ]:
vehicles, n_year_fixed= fix_year_manufactured(vehicles)
vehicles= add_vehicle_age_band(vehicles)
vehicles, n_capacity_filled= fix_capacity(vehicles)


Set 15 out-of-range year_manufactured values to null
Derived vehicle_age_band from year_manufactured
Filled 0 null capacity values using per-vehicle_type median


## Cleaning  the Stops File


In [ ]:
def fix_stop_name(df):
    df= df.copy()
    df["stop_name"] = df["stop_name"].str.strip().str.title()
    print("Trimmed whitespaces and title-cased stop_name")
    return df
stops= fix_stop_name(stops)


Trimmed whitespaces and title-cased stop_name


In [ ]:
def fix_has_shelter(df):
    df= df.copy()
    mapping= {
        "n": False, "0": False,
        "1": True, "y": True,
    }
    df["has_shelter"]= (
        df["has_shelter"].astype(str).str.strip().str.lower().map(mapping)
    )
    print("Normalised has_shelter to boolean")
    return df 
stops= fix_has_shelter(stops)

Normalised has_shelter to boolean


In [ ]:
print(stops["has_shelter"].unique())
print(stops["has_shelter"].isna().sum())

[False  True]
0


In [ ]:
def check_referential_integrity(validations, routes, vehicles, stops):
    validations= validations.copy()

    valid_routes= set(routes["route_id"])
    valid_vehicles= set(vehicles["vehicle_id"])
    valid_stops = set(stops["stop_id"])

    bad_route= -validations["route_id"].isin(valid_routes)
    bad_vehicle= -validations["vehicle_id"].isin(valid_vehicles)
    bad_stop= - validations["stop_id"].isin(valid_stops)

    bad_mask = bad_route | bad_vehicle | bad_stop
    rejects = validations[bad_mask].copy()

    def get_reason(row_bad_route, row_bad_vehicle, row_bad_stop):
        reasons = []
        if row_bad_route:
            reasons.append("unknown route_id")
        if row_bad_vehicle:
            reasons.append("unknown vehicle_id")
        if row_bad_stop:
            reasons.append("Unknown stop_id")
        return "; ".join(reasons)
    
    rejects["reject_reason"]=[
        get_reason(r, v, s)
        for r, v, s in zip(bad_route[bad_mask], bad_vehicle[bad_mask], bad_stop[bad_mask])
    ]

    clean= validations[-bad_mask].copy()

    print(f"Dropped {bad_mask.sum()} validations with broken route/vehicle/stop references")
    print(f"- unknown route_id: {bad_route.sum()}")
    print(f"- unknown vehicle_id: {bad_vehicle.sum()}")
    print(f"- unknown stop_id: {bad_stop.sum()}")

    return clean, rejects
validations, ref_integrity_rejects= check_referential_integrity(validations, routes, vehicles, stops)

Dropped 11782 validations with broken route/vehicle/stop references
- unknown route_id: 11763
- unknown vehicle_id: 13
- unknown stop_id: 13


In [ ]:
import os
os.makedirs("data/rejects", exist_ok=True)

In [ ]:
validations["route_id"]= validations["route_id"].str.strip()
validations["vehicle_id"]= validations["vehicle_id"].str.strip()
validations["stop_id"]= validations["stop_id"].str.strip()


In [ ]:
validations, ref_integrity_rejects= check_referential_integrity(validations, routes, vehicles, stops)

Dropped 0 validations with broken route/vehicle/stop references
- unknown route_id: 0
- unknown vehicle_id: 0
- unknown stop_id: 0


In [ ]:
ref_integrity_rejects.to_csv("data/rejects/validations_referential_integrity_rejects.csv", index= False)

In [ ]:
print(validations.shape)
print(ref_integrity_rejects.shape)

(67407, 11)
(0, 12)


In [ ]:
print(validations["route_id"].apply(repr).head(10))
print(routes["route_id"].apply(repr).head(10))

      

1     'R058'
2     'R043'
4     'R003'
5     'R013'
6     'R031'
8     'R019'
10    'R044'
11    'R049'
12    'R026'
13    'R054'
Name: route_id, dtype: str
0    'R001'
1    'R002'
2    'R003'
3    'R004'
4    'R005'
5    'R006'
6    'R007'
7    'R008'
8    'R009'
9    'R010'
Name: route_id, dtype: str


In [ ]:
common= set(validations["route_id"]) & set(routes["route_id"])
print(len(common))
print(validations["route_id"].nunique(), routes["route_id"].nunique())

60
60 60


In [ ]:
print(validations.shape)
print(ref_integrity_rejects.shape)

(67407, 11)
(0, 12)


In [ ]:
common_v = set(validations["vehicle_id"]) & set(vehicles["vehicle_id"])
print(len(common_v), validations["vehicle_id"].nunique(), vehicles["vehicle_id"].nunique())

common_v = set(validations["stop_id"]) & set(stops["stop_id"])
print(len(common_v), validations["stop_id"].nunique(), stops["stop_id"].nunique())



180 180 180
400 400 400


In [ ]:
ref_integrity_rejects.to_csv("data/rejects/validations_referental_integrity_rejects.csv", index=False)

### Derived columns


In [ ]:
import numpy as np

In [ ]:
def add_date_parts(df):
    df=df.copy()
    df["date"]= df["validation_ts"].dt.date
    df["year"]= df["validation_ts"].dt.year
    df["month"]= df["validation_ts"].dt.month
    df["year_month"]= df["validation_ts"].dt.to_period("M").astype(str)
    df["quarter"]= df["validation_ts"].dt.quarter

    print("Derived date, year, month, year_month, quarter from validations_ts")
    return df

validations = add_date_parts(validations)

Derived date, year, month, year_month, quarter from validations_ts


In [ ]:
def add_hour(df):
    df=df.copy()
    df["hour"]=df["validation_ts"].dt.hour
    print("Derived hour from validation_ts")
    return df
validations= add_hour(validations)

Derived hour from validation_ts


In [ ]:
def add_day_of_week(df):
    df= df.copy()
    df["day_of_week"]= df["validation_ts"].dt.day_name()
    df["is_weekend"]= df["validation_ts"].dt.dayofweek >= 5
    print("Derived day_of_week and is_weekend from validation_ts")
    return df
validations= add_day_of_week(validations)

Derived day_of_week and is_weekend from validation_ts


In [ ]:
def add_time_period(df):
    df= df.copy()
    conditions= [
        df["hour"].between(4,6),
        df["hour"].between(7,9),
        df["hour"].between(10,15),
        df["hour"].between(16,18),
        df["hour"].between(19,23),
    ]
    choices= ["Early","AM Peak", "Midday", "PM Peak", "Evening"]
    df["time_period"]= np.select(conditions,choices, default="Night")

    print("Derived time_period from hour")
    return df

validations= add_time_period(validations)

Derived time_period from hour


In [ ]:
def add_revenue(df):
    df=df.copy()
    df["revenue"]= df["fare_amount"]*df["passenger_count"]
    print("Derived revenue as fare-amount * passenger_count")
    return df
validations= add_revenue(validations)

Derived revenue as fare-amount * passenger_count


In [ ]:
validations[["validation_ts", "date", "year", "month", "year_month", "quarter","hour", "day_of_week", "is_weekend","time_period", "revenue"]].head()

,validation_ts,date,year,month,year_month,quarter,hour,day_of_week,is_weekend,time_period,revenue
1,2024-06-06 08:48:00,2024-06-06,2024,6,2024-06,2,8,Thursday,False,AM Peak,2.48
2,2024-03-06 13:15:00,2024-03-06,2024,3,2024-03,1,13,Wednesday,False,Midday,2.85
4,2024-06-14 18:35:27,2024-06-14,2024,6,2024-06,2,18,Friday,False,PM Peak,1.34
5,2024-02-16 18:30:59,2024-02-16,2024,2,2024-02,1,18,Friday,False,PM Peak,2.47
6,2024-03-03 08:41:00,2024-03-03,2024,3,2024-03,1,8,Sunday,True,AM Peak,0.13


In [ ]:
import os 
os.makedirs("data/interim", exist_ok=True)
os.makedirs("data/rejects", exist_ok=True)

validations.to_csv("data/interim/validations.csv", index=False)
routes.to_csv("data/interim/routes.csv", index=False)
vehicles.to_csv("data/interim/vehicles.csv", index=False)
stops.to_csv("data/interim/stops.csv", index=False)

print("Saved cleaned frames to data/interim")

Saved cleaned frames to data/interim


In [ ]:
ref_integrity_rejects.to_csv("data/rejects/validations_referential_integrity_rejects.csv", index=False)
print("Saved rejected rows to data/rejects")

Saved rejected rows to data/rejects
